# Декораторы и классы

## Классы как декораторы

### Шаг 1: Объясняем, что декоратором может быть любой вызываемый объект (callable)

До сих пор мы исходили из предположения, что декоратор - это всегда функция, которая оборачивает другую <b>функцию</b>. Это самый частый, но не единственный случай.

На самом деле, синтаксис @ в Python более гибкий и мощный. Декоратором может бть не только функция, но и <b>любой вызываемый объект (callable object)</b>.

#### Что такое "вызываемый объект"?

"Вызываемый объект" - это любой объект в Python, который можно "вызвать", используя круглые скобки (), как будто это функция.

Чтобы проверить, является ли объект вызываемым, можно использовать встроенную функцию callable().

Давайте посмотрим на примеры:

In [ ]:
# Функции, очевидно, вызываемые
def my_func():
    pass

print(f"Функция my_func вызываемая? {callable(my_func)}") # Вывод: True

# Числа или списки - нет
my_number = 42
my_list = [1, 2, 3]
print(f"Число my_number вызываемое? {callable(my_number)}") # Вывод: False
print(f"Список my_list вызываемый? {callable(my_list)}")   # Вывод: False

Функция my_func вызываемая? True
Число my_number вызываемое? False
Список my_list вызываемый? False


#### Как сделать экземпляр класса вызываемым?

А вот и ключ к нашей теме. Чтобы экземпляр вашего класса стал вызываемым, вам нужно определить в этом классе специальный, "магический" метод - <b>\_\_call__()</b> (два подчеркивания до и после).

Когда вы определяете этот метод, вы фактически говорите Python: "Вот что нужно делать, когда кто-то попытается "вызвать" экземпляр этого класса".

In [4]:
class Greeter:
    def __init__(self, greeting):
        self.greeting = greeting
        print("-> Экземпляр Greeter создан!")

    # Этот метод делает экземпляры класса вызываемыми
    def __call__(self, name):
        print(f"-> Метод __call__ вызван!")
        return f"{self.greeting}, {name}!"

# Создаем экземпляр класса
english_greeter = Greeter("Hello")

# Проверяем, является ли наш ЭКЗЕМПЛЯР вызываемым
print(f"Экземпляр english_greeter вызываемый? {callable(english_greeter)}")

# А теперь "вызываем" его, как будто это функция!
message = english_greeter("Alice")
print(message)

-> Экземпляр Greeter создан!
Экземпляр english_greeter вызываемый? True
-> Метод __call__ вызван!
Hello, Alice!


Это сработало! english_greeter('Alice') - это просто синтаксический сахар для english_greeter.\_\_call__('Alice').

#### Как это связано с декораторами?

Теперь вспомним, что делает синтаксис @:

In [7]:
# @my_decorator
# def some_function():
#     pass

Это эквивалентно:

In [8]:
# some_function = my_decorator(some_function)

Если my_decorator - это функция, все понятно. Но если my_decorator - это <b>экземпляр класса</b>, то эта запись тоже будет работать, при условии, что этот экземпляр - вызываемый!

#### Зачем это нужно? Главное преимущество - хранение состояния

Зачем усложнять и использовать классы вместо простых функций-декораторов? Главный ответ: <b>для хранения состояния</b>.

- В функции-декораторе, чтобы что-то "запомнить" между вызовами (как в нашем счетчике), нам приходилось использовать замыкания и nonlocal.

- В классе же мы можем легко хранить любое состояние в атрибутах экземпляра (например, self.call_count). Это часто бывает более читаемо и удобно для управления сложной логикой.

#### Итог

Декоратор - это не обязательно функция. Это может быть <b>любой вызываемый объект</b>. Экземпляры классов становятся вызываемыми, если в них определен метод \_\_call__.

### Шаг 2: Показываем, как создать класс-декоратор, определив \_\_init__ и \_\_call__

Чтобы класс мог работать как декоратор, нам нужно использовать два его магических метода для разных целей:

1. <b>\_\_init__(self, func)</b>: Конструктор класса. Он будет вызываться <b>один раз</b>, в момент, когда мы "украшаем" функцию. Его задача - принять декорируемую функцию (func) и сохранить ее для будущего использования, например, в атрибуте self.func.

2. <b>\_\_call__(self, *args, **kwargs)</b>: Метод вызова. Он будет выполняться <b>каждый раз</b>, когда мы вызываем нашу уже декорированную функцию. Именно здесь мы реализуем всю логику "обертки": код "до", вызов сохраненной self.func и код "после".

#### Давайте перепишем наш самый первый simple_decorator в виде класса

##### Версия на функциях (для сравнения):

In [10]:
def simple_function_decorator(func):
    def wrapper(*args, **kwargs):
        print("[FUNC_DECO]: Перед вызовом")
        result = func(*args, **kwargs)
        print("[FUNC_DECO]: После вызова")
        return result
    return wrapper

##### Новая версия на классе:

In [11]:
class SimpleClassDecorator:
    # 1. __init__ принимает и сохраняет исходную функцию
    def __init__(self, func):
        print(f"-> __init__ вызван для функции '{func.__name__}'")
        self.func = func # Сохраняем функцию как атрибут

    # 2. __call__ реализует логику обертки
    def __call__(self, *args, **kwargs):
        print(f"-> __call__ вызван. Оборачиваем вызов {self.func.__name__}")
        
        # --- Логика "ДО" ---
        print("[CLASS_DECO]: Перед вызовом")
        
        # Вызываем сохраненную оригинальную функцию
        result = self.func(*args, **kwargs)
        
        # --- Логика "ПОСЛЕ" ---
        print("[CLASS_DECO]: После вызова")
        
        return result

#### Как это работает "под капотом"?

Давайте посмотрим, что происходит, когда Python встречает этот код:

In [12]:
@SimpleClassDecorator
def say_hello(name):
    print(f"    Привет, {name}!")

# Вызываем декорированную функцию
say_hello("Мир")

-> __init__ вызван для функции 'say_hello'
-> __call__ вызван. Оборачиваем вызов say_hello
[CLASS_DECO]: Перед вызовом
    Привет, Мир!
[CLASS_DECO]: После вызова


1. <b>Применение декоратора</b>: Python видит @SimpleClassDecorator. Он понимает, что это класс, и выполняет эквивалент следующей операции: <i>say_hello = SimpleClassDecorator(say_hello)</i>

2. <b>Вызов \_\_init__</b>: Эта операция (SimpleClassDecorator(say_hello)) создает <b>экземпляр</b> нашего класса. При этом автоматически вызывается его конструктор \_\_init__.

    - В self передается сам создаваемый экземпляр.

    - В func передается наша исходная функция say_hello.

    - Конструктор сохраняет эту функцию в self.func и печатает сообщение -> \_\_init__ вызыван...

    - Теперь имя say_hello в нашей программе ссылается на этот <b>созданный экземпляр</b>.

3. <b>Вызов функции</b>: Позже, когда мы пишем say_hello("Мир"), мы на самом деле <b>"вызываем" экземпляр</b>, который хранится в переменной say_hello.

    - Python видит, что мы пытаемся вызвать объект, и автоматически вызывает его метод \_\_call__.

    - Аргументы ("Мир",) передаются в *args.

    - Метод \_\_call__ выполняет нашу логику "до" и "после" и вызывает оригинальную функцию, которую он ранее сохранил в self.func.

#### Сохранение метаданных с functools.update_wrapper

Кстати, проблема с потерей метаданных (\_\_name__, \_\_doc__) актуальная и для классовых декораторов. Решение для нее очень похоже на @functools.wraps. Внутри \_\_init__ нужно использовать функцию functools.update_wrapper().

In [13]:
from functools import update_wrapper

class SimpleClassDecoratorWithMetadata:
    def __init__(self, func):
        self.func = func
        update_wrapper(self, func) # Копируем метаданные из func в self (экземпляр)

    def __call__(self, *args, **kwargs):
        # ... логика ...
        return self.func(*args, **kwargs)

#### Итог

Мы можем реализовать логику декоратора с помощью класса, разделив обязанности между методами \_\_init__ и \_\_call__.

- <b>\_\_init__</b> - для "настройки" и <b>сохранения</b> декорируемой функции (выполняется один раз при декорировании).

- <b>\_\_call__</b> - для <b>выполнения</b> логики обертки (выполняется каждый раз при вызове).

Эта структура становится особенно полезной, когда нам нужно хранить состояние между вызовами, чем мы и займемся в следующем шаге.

### Шаг 3: Обсуждаем главное преимущество: хранение состояния в атрибутах экземпляра

До сих пор мы могли бы сказать, что классовый декоратор - это просто более громоздкий и "модный" способ написать то, что можно сделать с помощью вложенных функций: И до определенного момента это так.

Но у классов есть одно фундаментальное преимущество, которое делает их незаменимыми для определенного типа задач - это <b>простое и естественное хранение состояния (state)</b>.

#### Что такое "состояние" в контексте декоратора?

"Состояние" - это любая информация, которую декоратор должен "помнить" <b>между</b> разными вызовами декорируемой функции.

Давайте вспомним наш пример с функцией-счетчиком из Модуля 1, который мы реализовывали через замыкание:

In [14]:
def function_counter_decorator(func):
    count = 0
    def wrapper(*args, **kwargs):
        nonlocal count # <--- Приходится использовать 'nonlocal'
        count += 1
        print(f"Функция '{func.__name__}' была вызвана {count} раз.")
        return func(*args, **kwargs)
    return wrapper

Это работает, но использование nonlocal не всегда очевидно, и если логика состояния усложняется (нужно хранить не только счетчик, а 5-6 разных переменных), код становится запутанным.

#### Реализация того же счетчика с помощью класса

С классом та же самая задача решается гораздо более естественно и читаемо. Мы просто храним счетчик в атрибуте экземпляра self.count.

In [15]:
from functools import update_wrapper

class StatefulCounterDecorator:
    def __init__(self, func):
        self.func = func
        # 1. Инициализируем наше состояние в конструкторе
        self.call_count = 0
        update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        # 2. Обращаемся к состоянию и изменяем его
        self.call_count += 1
        print(f"Функция '{self.func.__name__}' была вызвана {self.call_count} раз.")
        
        # Вызываем оригинальную функцию
        return self.func(*args, **kwargs)

# --- Демонстрация ---

@StatefulCounterDecorator
def some_function():
    print("    ... выполняется тело some_function ...")

print("--- Начинаем вызовы ---")
some_function()
some_function()
some_function()

--- Начинаем вызовы ---
Функция 'some_function' была вызвана 1 раз.
    ... выполняется тело some_function ...
Функция 'some_function' была вызвана 2 раз.
    ... выполняется тело some_function ...
Функция 'some_function' была вызвана 3 раз.
    ... выполняется тело some_function ...


<b>Анализ</b>:

1. Когда мы декорируем some_function, создается <b>один-единственный экземпляр</b> StatefulCounterDecorator. В его конструкторе \_\_init__ создается атрибут self.call_count = 0.

2. При <b>каждом</b> вызове some_function() на самом деле вызывается метод \_\_call__ <b>этого же самого экземпляра</b>.

3. Соответственно, \_\_call__ имеет доступ к <b>одному и тому же</b> атрибуту self.call_count и может его изменять.

#### Преимущества классового подхода для хранения состояния:

- <b>Явное и понятное</b>: Любой, кто знаком с ООП, сразу поймет, что self.call_count - это состояние, принадлежащее этому конкретному декорирующему объекту. Это более читаемо, чем замыкания и nonlocal.

- <b>Гибкость</b>: Мы можем хранить любое количество переменных состояния. Можно добавить self.last_call_time, self.results_history = [] и т.д., и вся логика останется инкапсулированной внутри одного класса.

- <b>Инкапсуляция</b>: Вся логика, связанная с управлением состоянием, находится внутри методов класса, а не размазана по вложенным функциям.

#### Когда это особенно полезно?

- <b>Кэширование</b>: Наш самописный декоратор @cache можно было бы элегантно реализовать через класс, храня memo_cache в self.cache.

- <b>Ограничение частоты вызовов (Throttling / Rate Limiting)</b>: Декоратор, который позволяет вызывать функцию не чаще, чем раз в N секунд. Ему нужно хранить время последнего успешного вызова (self.last_call_time).

- <b>Управление ресурсами</b>: Декоратор, который открывает соединение с базой данных при первом вызове и закрывает его после N вызовов. Ему нужно хранить и счетчик, и объект соединения.

#### Итог

Главная причина использовать классы для создания декораторов - это необходимость <b>хранить и управлять состоянием между вызовами</b> декорируемой функции.

Классы предоставляют для этого естественный, читаемый и мощный механизм через атрибуты экземпляра (self.что-то), что часто является лучшей альтернативой сложным замыканиям.

### Задачи

#### Задача 1: Простой декоратор-класс

<b>Условие задачи</b>:

Напишите <b>класс-декоратор</b> DecoratorClass.
Он должен работать так же, как и наши первые функциональные декораторы:
1. В \_\_init__ он должен принимать и сохранять декорируемую функцию.
2. В \_\_call__ он должен печатать "--- Before ---", вызывать сохраненную функцию, а затем печатать "--- After ---".

In [16]:
from functools import update_wrapper

class DecoratorClass:
    def __init__(self, func):
        self.func = func
        update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        print('--- Before ---')
        result = self.func(*args, **kwargs)
        print('--- After ---')
        return result

#### Задача 2: Класс-счетчик вызовов

<b>Условие задачи</b>:

Напишите <b>класс-декоратор</b> CallCounter.

Этот декоратор должен считать, сколько раз была вызвана декорируемая им функция. При каждом вызове он должен печатать строку в формате "Function 'имя_функции' called {N} times.", где N — номер текущего вызова.

Подсказка: инициализируйте счетчик в \_\_init__ и увеличивайте его в \_\_call__.

In [ ]:
from functools import update_wrapper

class CallCounter:
    def __init__(self, func):
        self.count = 0
        self.func = func
        update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        result = self.func(*args, **kwargs)
        self.count += 1
        print(f'Function {self.func.__name__!r} called {self.count} times.')
        return result

#### Задача 3: Класс-декоратор с лимитом вызовов

<b>Условие задачи</b>:

Напишите <b>класс-декоратор</b> LimitCalls.

Этот декоратор должен позволить вызвать декорируемую функцию <b>только 2 раза</b>. При третьем и последующих вызовах он должен выбрасывать исключение ValueError с сообщением "Function call limit exceeded".

In [19]:
from functools import update_wrapper

class LimitCalls:
    def __init__(self, func):
        self.count = 0
        self.func = func
        update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        self.count += 1
        
        if self.count > 2:
            raise ValueError('Function call limit exceeded')
        
        return self.func(*args, **kwargs)

#### Задача 4: Декоратор, запоминающий последний результат

<b>Условие задачи</b>:

Напишите <b>класс-декоратор</b> LastResult.

Этот декоратор должен хранить результат с<b>амого последнего</b> вызова функции.
1. В \_\_init__ создайте атрибут для хранения последнего результата (например, self.last_result = None).
2. В \_\_call__ вызовите функцию, сохраните ее результат в этот атрибут и верните его.

In [20]:
from functools import update_wrapper

class LastResult:
    def __init__(self, func):
        self.last_result = None
        self.func = func
        update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        self.last_result = self.func(*args, **kwargs)
        return self.last_result

#### Задача 5: Класс-декоратор с сохранением метаданных

<b>Условие задачи</b>:

Напишите <b>класс-декоратор</b> ProperDecorator, который правильно сохраняет метаданные (\_\_name__ и \_\_doc__) декорируемой функции.

Логика самого декоратора проста: он должен просто вызывать исходную функцию.

Подсказка: используйте functools.update_wrapper(self, func) в конструкторе \_\_init__.

In [27]:
from functools import update_wrapper

class ProperDecorator:
    def __init__(self, func):
        metadata = ('__name__', '__doc__')
        self.func = func
        update_wrapper(self, func, assigned=metadata)

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

## Декорирование методов класса

### Шаг 1: Особенности применения обычных декораторов к методам класса

Мы научились создавать декораторы и применять их к обычным "свободным" функциям. Теперь давайте посмотрим, что произойдет, если мы применим те же самые декораторы к методам внутри класса.

Синтаксис остается абсолютно таким же - мы просто ставим @имя_декоратора над определением метода.

In [28]:
def my_simple_logger(func):
    """Простой логгер для демонстрации."""
    def wrapper(*args, **kwargs):
        print(f"--- Вызывается метод: '{func.__name__}' с аргументами {args} и {kwargs}")
        result = func(*args, **kwargs)
        return result
    return wrapper

class Calculator:
    def __init__(self, brand):
        self.brand = brand

    @my_simple_logger
    def add(self, a, b):
        """Складывает два числа."""
        return a + b

    @my_simple_logger
    def subtract(self, a, b):
        """Вычитает одно число из другого."""
        return a - b

# --- Создаем экземпляр и вызываем методы ---
calc = Calculator("Casio")
calc.add(10, 5)
calc.subtract(100, 20)

--- Вызывается метод: 'add' с аргументами (<__main__.Calculator object at 0x0000045035111B10>, 10, 5) и {}
--- Вызывается метод: 'subtract' с аргументами (<__main__.Calculator object at 0x0000045035111B10>, 100, 20) и {}


80

#### Особенность №1: self становится первым аргументом

Посмотрите внимательно на вывод! Когда мы вызывали calc.add(10, 5), наш декоратор "увидел" <b>три</b> позиционных аргумента, а не два.

args = (<\_\_main__.Calculator object at 0x...>,  10, 5)

##### Почему так произошло?

Нужно вспомнить, как работают методы в Python. Вызов calc.add(10, 5) - это синтаксический сахар для Calculator.add(calc, 10, 5). Python автоматически передает экземпляр класса (calc) в качестве <b>первого</b> позиционного аргумента в метод. Этот аргумент мы по соглашению называем self.

Наш универсальный декоратор, написанный с использованием \*args и \*\*kwargs, <b>справляется с этой ситуацией без проблем</b>! Он просто "упаковывает" self в args вместе с остальными аргументами и так же "распаковывает" их при вызове func(*args, **kwargs).

<b>Вывод</b>: Обычные, правильно написанные (с *args, **kwargs) декораторы-функции <b>прекрасно работают</b> для декорирования методов экземпляра.

#### Особенность №2: Доступ к self внутри декоратора

Хорошо, декоратор работает. Но что, если мы хотим сделать его "умнее"? Что, если декоратор должен <b>использовать</b> какие-то данные из самого экземпляра?

Например, мы хотим, чтобы наш логгер выводил не только имя метода, но и бренд калькулятора, который хранится в self.brand.

Мы можем легко это сделать, ведь self - это просто первый элемент в кортеже args!

In [29]:
def instance_aware_logger(func):
    """Логгер, который умеет 'заглядывать' в self."""
    def wrapper(*args, **kwargs):
        # self - это всегда первый позиционный аргумент для метода экземпляра
        self_instance = args[0]
        
        # Теперь мы можем получить доступ к атрибутам экземпляра
        print(f"--- [Бренд: {self_instance.brand}] Вызывается метод: '{func.__name__}'")
        
        result = func(*args, **kwargs)
        return result
    return wrapper

class Calculator:
    def __init__(self, brand):
        self.brand = brand

    @instance_aware_logger
    def add(self, a, b):
        return a + b

# --- Тестируем "умный" логгер ---
calc = Calculator("Texas Instruments")
calc.add(25, 35)

--- [Бренд: Texas Instruments] Вызывается метод: 'add'


60

Это сработало! Наш декоратор смог получить доступ к self.brand и использовать его в своем сообщении.

#### Итог

1. Обычные декораторы-функции, написанные с *args и **kwargs, без проблем можно применять к методам класса.

2. При декорировании метода экземпляра, сам экземпляр (self) автоматически передается в декоратор как <b>первый позиционный аргумент</b> (args[0]).

3. Это позволяет нам создавать "умные" декораторы, которые могут читать (и даже изменять) состояние объекта, к методу которого они применяются. Это открывает путь к реализации сложной логики, например, для проверки прав доступа, кэширования на уровне экземпляра и т.д.

### Шаг 2: self как неявный первый аргумент в обертке

Когда мы работаем с декораторами и методами классов, самое важное и поначалу неочевидное - это понять, <b>откуда берется и что из себя представляет</b> первый аргумент, который "прилетает" в нашу функцию-обертку.

Давайте еще раз посмотрим на этот код, но теперь сфокусируемся на моменте вызова.

In [30]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"WRAPPER ПОЛУЧИЛ: args={args}, kwargs={kwargs}")
        # ...
        return func(*args, **kwargs)
    return wrapper

class MyClass:
    @my_decorator
    def my_method(self, x, y):
        print("    (Выполняется тело my_method)")

# 1. Создаем экземпляр
instance = MyClass()

# 2. Вызываем метод
instance.my_method(10, 20)

WRAPPER ПОЛУЧИЛ: args=(<__main__.MyClass object at 0x0000045035111E10>, 10, 20), kwargs={}
    (Выполняется тело my_method)


#### Разбираем магию вызова instance.my_method(10, 20)

В Python этот, казалось бы, простой вызов на самом деле является "синтаксическим сахаром" для более сложного процесса, который происходит "под капотом". Этот процесс называется <b>связыванием метода (method binding)</b>.

Вот что происходит по шагам:

1. <b>Поиск атрибута</b>: Python ищет атрибут my_method у объекта instance.

2. <b>Обнаружение декоратора</b>: Он находит не саму функцию my_method, а <b>обертку wrapper</b>, которой она была заменена в момент создания класса.

3. <b>Связывание</b>: Python видит, что эта wrapper (которая ведет себя как функция) была найдена через экземпляр instance. Поэтом он создает временный объект, называемый <b>"связанный метод" (bound method)</b>. Этот объект "помнит" и саму функцию (wrapper), и экземпляр (instance).

4. <b>Трансформация вызова</b>: Теперь Python преобразует наш вызова instance.my_method(10, 20) в следующий: он вызывает функцию wrapper, автоматически подставляя instance в качетстве <b>самого первого аргумента</b>: wrapper(instance, 10, 20).

Именно поэтому, когда мы смотрим на print внутри нашего декоратора, мы видим: WRAPPER ПОЛУЧИЛ: args=(<\_\_main__.MyClass object at 0x...>, 10, 20), kwargs={}

#### self - это просто соглашение об имени

Внутри определения метода my_method(self, x, y) первый параметр мы называем self. Это всего лишь <b>соглашение</b>. Мы могли бы назвать его this, obj или как угодно еще.

Важно то, что Python <b>всегда</b> передает экземпляр класса как первый позиционный аругмент в любой метод экземпляра. И поскольку наш декоратор wrapper(*args, **kwargs) просто "пропускает" через себя все аргументы, self естественным образом оказывается на первом месте в кортеже args.

#### Визуализация потока данных:

In [ ]:
# Вызов в коде            |     Трансформация Python'ом     | Что "видит" декоратор
# ------------------------|---------------------------------|---------------------------------
#                         |                                 | def wrapper(*args, **kwargs):
# instance.my_method(10)  |  -> wrapper(instance, 10)   ->  |   args = (instance, 10)
#                         |                                 |   kwargs = {}
# ------------------------|---------------------------------|---------------------------------
#                         |                                 | def wrapper(*args, **kwargs):
# instance.my_method(y=20)|  -> wrapper(instance, y=20) ->  |   args = (instance,)
#                         |                                 |   kwargs = {'y': 20}
# ------------------------|---------------------------------|---------------------------------

#### Итог

При декорировании метода экземпляра его self не исчезает и не требует какой-то специальной обработки. Он просто становится <b>первым элементом в кортеже *args</b>, который получает ваша функция-обертка.

Понимание этого простого факта (self == args[0]) - это ключ к созданию декораторов, которые могут взаимодействовать с состоянием и поведением объекта, которому принадлежит декорируемый метод.

#### Шаг 3: Показываем, как получить доступ к self и его атрибутам внутри декоратора

Итак, мы установили, что при вызове декорированного метода self (экземпляр класса) оказывается первым элементом в кортеже args. Это знание открывает нам огромные возможности! Мы можем написать декоратор, логика которого будет зависеть от состояния объекта.

#### Задача: Декоратор для проверки прав доступа

Давайте представим, что мы пишем систему, где есть пользователи с разными ролями. Некоторые действия, такие как удаление записей, должны быть доступны только пользователям с ролью "администратор".

Мы можем реализовать эту проверку с помощью декоратора.

##### 1. Сначала создадим простой класс User:

In [1]:
class User:
    def __init__(self, name, role):
        self.name = name
        self.role = role

    def __str__(self):
        return f"User(name='{self.name}', role='{self.role}')"

##### 2. Теперь напишем декоратор @requires_role:

Это будет <b>декоратор с аргументами</b> (фабрика), так как нам нужно указать, какая именно роль требуется для выполнения метода.

In [2]:
from functools import wraps

def requires_role(required_role):
    """
    Фабрика декораторов. Проверяет, что у пользователя,
    вызывающего метод, есть необходимая роль.
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            # 1. Получаем доступ к 'self'
            #    args[0] - это экземпляр, на котором вызван метод
            #    В нашем случае это будет экземпляр класса AdminPanel.
            instance = args[0]
            
            # 2. Получаем доступ к атрибуту 'self.current_user'
            #    Предполагаем, что в декорируемом классе есть такой атрибут.
            current_user = instance.current_user
            
            print(f"[ПРОВЕРКА]: Требуемая роль: '{required_role}'. Текущий пользователь: {current_user}")
            
            # 3. Реализуем сложную логику: сравниваем роли
            if not current_user or current_user.role != required_role:
                raise PermissionError(
                    f"Доступ запрещен. Для выполнения '{func.__name__}' "
                    f"требуется роль '{required_role}'."
                )
            
            # 4. Если проверка пройдена, выполняем исходный метод
            print("[ПРОВЕРКА]: Доступ разрешен.")
            return func(*args, **kwargs)
            
        return wrapper
    return decorator

##### 3. Создадим класс AdminPanel, где применим наш декоратор:

Этот класс будет имитировать панель управления, которая хранит информацию о текущем вошедшем в систему пользователе.

In [3]:
class AdminPanel:
    def __init__(self, current_user: User):
        # Этот атрибут декоратор будет "читать"
        self.current_user = current_user

    @requires_role("admin")
    def delete_everything(self):
        """Опасный метод, требующий прав администратора."""
        print("    >>> Все данные успешно удалены! <<<")
        
    @requires_role("viewer")
    def view_dashboard(self):
        """Метод, доступный для просмотра."""
        print("    >>> Отображение панели мониторинга... <<<")

##### 4. Тестируем нашу систему:

In [4]:
# Создаем двух пользователей
admin_user = User(name="Alice", role="admin")
simple_user = User(name="Bob", role="viewer")

# --- Сценарий 1: Администратор пытается удалить данные (успех) ---
print("--- Сценарий 1: Администратор выполняет опасное действие ---")
panel_for_admin = AdminPanel(current_user=admin_user)
try:
    panel_for_admin.delete_everything()
except PermissionError as e:
    print(f"ОШИБКА: {e}")

# --- Сценарий 2: Обычный пользователь пытается удалить данные (провал) ---
print("\n--- Сценарий 2: Обычный пользователь выполняет опасное действие ---")
panel_for_user = AdminPanel(current_user=simple_user)
try:
    panel_for_user.delete_everything()
except PermissionError as e:
    print(f"ОШИБКА: {e}")

--- Сценарий 1: Администратор выполняет опасное действие ---
[ПРОВЕРКА]: Требуемая роль: 'admin'. Текущий пользователь: User(name='Alice', role='admin')
[ПРОВЕРКА]: Доступ разрешен.
    >>> Все данные успешно удалены! <<<

--- Сценарий 2: Обычный пользователь выполняет опасное действие ---
[ПРОВЕРКА]: Требуемая роль: 'admin'. Текущий пользователь: User(name='Bob', role='viewer')
ОШИБКА: Доступ запрещен. Для выполнения 'delete_everything' требуется роль 'admin'.


##### Анализ:

Наш декоратор @reauires_role успешно справился с задачей!

- Он "заглянул" внутр экземпляра AdminPanel через self (который был args[0]).

- Он получил доступ к атрибуту self.current_user.

- Он смог прочитать данные из этого атрибута (current_user.role).

- На основе этих данных он принял решение: либо разрешить выполнение метода, либо выбросить исключение PermissionError.

#### Итог

Доступ к self внутри декоратора - это чрезвычайно мощный механизм. Он позволяет декораторам выйти за рамки простой обертки и реализовать сложную, контекстно-зависимую бизнес-логику. Декоратор может стать полноценным участником взаимодействия с объектом, проверяя его состояние и динамически изменяя поведение его методов.

### Задачи

#### Задача 1: Логгер, использующий self

<b>Условие задачи</b>:

Напишите декоратор log_with_instance_name.

Декоратор будет применяться к методу класса, у которого есть атрибут name. Декоратор должен перед вызовом метода печатать лог в формате [ИМЯ_ЭКЗЕМПЛЯРА]: Calling 'имя_метода'.

Подсказка: экземпляр класса — это args[0], а его имя — args[0].name.

In [6]:
from functools import wraps

def log_with_instance_name(func):
    @wraps(func)
    def wrapper(instance, *args, **kwargs):
        print(f'[{instance.name}]: Calling {func.__name__!r}')
        return func(instance, *args, **kwargs)
    return wrapper

#### Задача 2: Декоратор для проверки состояния

<b>Условие задачи</b>:

Напишите декоратор check_state(required_state).

Он будет применяться к методам класса, у которого есть атрибут state. Декоратор должен проверять, совпадает ли self.state с required_state.
- Если состояния совпадают, метод выполняется.
- Если нет, декоратор должен выбросить ValueError с сообщением "Invalid state".

In [15]:
from functools import wraps

def check_state(required_state):
    def decorator(func):

        @wraps(func)
        def wrapper(instance, *args, **kwargs):
            if getattr(instance, 'state', None) != required_state:
                raise ValueError('Invalid state')
            return func(instance, *args, **kwargs)

        return wrapper
    return decorator

#### Задача 3: Декоратор, изменяющий состояние

<b>Условие задачи</b>:

Напишите декоратор set_state_after.

Декоратор будет применяться к методам класса Worker, у которого есть атрибут status. После <b>успешного</b> выполнения декорируемого метода, декоратор должен изменить self.status на "completed".

In [16]:
from functools import wraps

def set_state_after(func):
    @wraps(func)
    def wrapper(instance, *args, **kwargs):
        result = func(instance, *args, **kwargs)
        instance.status = 'completed'
        return result
    return wrapper

#### Задача 4: Декоратор, вызывающий другой метод

<b>Условие задачи</b>:

Напишите декоратор audit.

Он будет применяться к методам класса Account, у которого есть метод add_audit_log(message). Декоратор audit должен после выполнения своей основной функции вызвать метод self.add_audit_log, передав в него сообщение вида "Method '{имя_метода}' was called".

In [17]:
from functools import wraps

def audit(func):
    @wraps(func)
    def wrapper(instance, *args, **kwargs):
        result = func(instance, *args, **kwargs)
        instance.add_audit_log(f'Method {func.__name__!r} was called')
        return result
    return wrapper

#### Задача 5: Декоратор-счетчик на уровне экземпляра

<b>Условие задачи</b>:

Напишите декоратор instance_level_counter.

Класс Button будет иметь атрибут-счетчик click_count. Декоратор, примененный к методу click, должен увеличивать этот счетчик (self.click_count) каждый раз при вызове.

<b>Важно</b>: каждый экземпляр Button должен иметь свой собственный, независимый счетчик.

In [18]:
from functools import wraps

def instance_level_counter(func):
    @wraps(func)
    def wrapper(instance, *args, **kwargs):
        instance.click_count += 1
        return func(instance, *args, **kwargs)
    return wrapper

## Встроенные декораторы: @staticmethod, @classmethod, @property

### Шаг 1: Вводим @staticmethod. Метод, не привязанный ни к экземпляру, ни к классу.

До сих пор мы рассматривали методы, которые всегда неявно принимают в качестве первого аргумента либо экземпляр класса (self), либо сам класс (cls). Но иногда внутри класса нам нужна функция, которая логически связана с этим классом, но для своей работы ей <b>не требуется доступ</b> ни к состоянию конкретного экземпляра (self), ни к самому классу (cls).

Такие функции можно было бы просто определить вне класса, но это "загрязняет" пространство имен и нарушает инкапсуляцию, ведь функция по смыслу относится именно к этому классу.

Для таких случаев в Python существует встроенный декоратор <b>@staticmethod</b>.

<b>Статический метод</b> - это, по сути, обычная функция, которая "живет" внутри класса. Она не получает никаких специальных первых аргументов.

#### Как это выглядит?

Давайте рассмотрим класс MathHelper, который будет содержать разные математические утилиты.

In [1]:
class MathHelper:
    def __init__(self, name):
        # Обычный метод имеет доступ к self.name
        self.name = name

    def triple(self, x):
        """Обычный метод экземпляра: умножает x на 3."""
        print(f"Вызван обычный метод экземпляра (self.name = '{self.name}')")
        return x * 3

    @staticmethod
    def add(a, b):
        """
        Статический метод. Ему НЕ НУЖЕН self.
        Он просто работает со своими аргументами a и b.
        """
        print("Вызван статический метод add()")
        return a + b

# --- Как его вызывать? ---

# 1. Вызов через класс
#    Это самый частый и логичный способ.
result_from_class = MathHelper.add(10, 5)
print(f"Результат вызова через класс: {result_from_class}")

# 2. Вызов через экземпляр
#    Это тоже работает, но смысла в этом мало,
#    так как self все равно не передается.
helper_instance = MathHelper("MyHelper")
result_from_instance = helper_instance.add(20, 30)
print(f"Результат вызова через экземпляр: {result_from_instance}")

# Сравните с вызовом обычного метода, который требует экземпляр
product = helper_instance.triple(4)
print(f"Результат вызова обычного метода: {product}")

Вызван статический метод add()
Результат вызова через класс: 15
Вызван статический метод add()
Результат вызова через экземпляр: 50
Вызван обычный метод экземпляра (self.name = 'MyHelper')
Результат вызова обычного метода: 12


<b>Анализ</b>:

- <b>Сигнатура</b>: Обратите внимание, что в def add(a, b) нет никакого self или cls. Это сигнатура обычной функции.

- <b>Вызов</b>: Мы можем вызвать add напрямую от имени класса: MathHelper.add(...). Нам не нужно для этого создавать экземпляр MathHelper().

- <b>Изоляция</b>: Внутри add мы не можем обратиться к self.name, потому что self туда просто не передается. Статический метод полностью изолирован от состояния экземпляра.

#### Когда использовать @staticmethod?

Используйте статические методы, когда вам нужна <b>функция-утилита</b>, которая:

1. <b>Логически связана</b> с классом (например, MathHelper.add - это логично).

2. <b>Не зависит</b> от состояния какого-либо конкретного экземпляра (self).

3. <b>Не зависит</b> от самого класса (cls), то есть не создает экземпляры этого класса и не обращается к его атрибутам.

<b>Примеры из реальной жизни</b>:

- Класс для работы с датами, а в нем - статический метод для валидации формата строки с датой.

- Класс, представляющий какой-то сложный объект, а в нем - статический метод для конвертации единиц измерения, используемых этим объектом.

#### Итог

Декоратор @staticmethod "отвязывает" метод от экземпляра и класса. Он превращает метод в обычную функцию, которая просто находится в пространстве имен класса для удобства и логической группировки. Это полезно для создания вспомогательных функций, которым не нужен доступ к состоянию объекта.

### Шаг 2: Разбираем @classmethod. Метод, работающий с классом.

Мы только что рассмотрели @staicmethod, который полностью "отвязывает" метод от класса. @classmethod же находится как бы посередине между обычным методом экземпляра и статическим методом.

<b>Классовый метод (class method)</b> - это метод, который привязан не к конкретному экземпляру, а к <b>классу в целом</b>. Вместо self он неявно получает в качестве первого аргумента сам <b>объект класса</b>. Этот аргумент по соглашению принято называть cls.

#### Как это выглядит?

Давай представим, что мы создаем класс для работы с датами.

In [2]:
class MyDate:
    def __init__(self, day, month, year):
        self.day = day
        self.month = month
        self.year = year

    def display(self):
        """Обычный метод, работает с self."""
        return f"{self.day:02d}-{self.month:02d}-{self.year}"

    @classmethod
    def from_string(cls, date_string):
        """
        Классовый метод. Он получает 'cls' (т.е. MyDate).
        Он работает как альтернативный конструктор.
        """
        print(f"Вызван классовый метод. cls = {cls.__name__}")
        # Разбираем строку формата "ДД-ММ-ГГГГ"
        day, month, year = map(int, date_string.split('-'))
        
        # Используем 'cls' для создания нового экземпляра этого же класса
        new_date_instance = cls(day, month, year)
        
        return new_date_instance

# --- Как его использовать? ---

# 1. Мы не можем создать объект напрямую, у нас есть только строка
date_str = "25-12-2025"

# 2. Вызываем классовый метод ПРЯМО ОТ КЛАССА
christmas_date = MyDate.from_string(date_str)

# 3. В результате мы получили готовый экземпляр!
print(f"Создан объект: {christmas_date}")
print(f"Его тип: {type(christmas_date)}")
print(f"Вызываем его обычный метод: {christmas_date.display()}")

Вызван классовый метод. cls = MyDate
Создан объект: <__main__.MyDate object at 0x000002AC926DFA90>
Его тип: <class '__main__.MyDate'>
Вызываем его обычный метод: 25-12-2025


<b>Анализ</b>:

- <b>Сигнатура</b>: Обратите внимание на def from_string(cls, date_string). Первый аргумент cls появился автоматически, так же, как self для обычных методов.

- <b>Доступ к классу</b>: Внутри from_string переменная cls является ссылкой на сам класс MyDate. Это позволяет нам сделать cls(day, month, year), что эквивалентно MyDate(day, month, year).

- <b>Назначение</b>: Мы использовали @classmethod для создания <b>альтернативного конструктора</b>. Наш освновной конструктор \_\_init__ принимает три числа, но нам было удобнее создавать объект из строки. @classmethod идеально решил эту задачу.

#### Зачем это нужно? Главное преимущество - работа с наследованием

Вы могли бы сказать: "А почему просто не написать MyDate(day, month, year) внутри from_string?". Это будет работать. Но @classmethod становится по-настоящему мощным при <b>наследовании</b>.

Давайте создадим дочерний класс:

In [3]:
class DateTime(MyDate): # Наследуемся от MyDate
    def display(self):
        # Переопределяем метод display
        return f"{self.year}/{self.month:02d}/{self.day:02d} (формат ISO)"

# Теперь ВЫЗЫВАЕМ ТОТ ЖЕ САМЫЙ КЛАССОВЫЙ МЕТОД, но от дочернего класса!
iso_date = DateTime.from_string("15-05-2026")

print(f"\nСоздан объект через дочерний класс: {iso_date}")
print(f"Его тип: {type(iso_date)}") # <-- ОБРАТИТЕ ВНИМАНИЕ НА ТИП!
print(f"Вызываем его переопределенный метод: {iso_date.display()}")

Вызван классовый метод. cls = DateTime

Создан объект через дочерний класс: <__main__.DateTime object at 0x000002AC926DFC10>
Его тип: <class '__main__.DateTime'>
Вызываем его переопределенный метод: 2026/05/15 (формат ISO)


<b>Магия в действии!</b> Когда мы вызвали DateTime.from_string(...), в cls пришел уже не MyDate, а DateTime! Поэтому cls(...) создало экземпляр именно дочернего класса DateTime. Наш классовый метод, написанный всего один раз в родительском классе, автоматически адаптировался для работы со всеми его наследниками.

#### Итог

Декоратор @classmethod "привязывает" метод к классу, а не к экземпляру, передавая сам класс в качестве первого аргумента cls. Его основное и самое мощное применение - это создание <b>альтернативных конструкторов</b>, которые правильно работают с наследованием.

### Шаг 3: Изучаем @property. Превращаем метод в атрибут

Мы рассмотрели методы, которые привязаны к классу (@classmethod) и которые не привязаны ни к чему (@staticmethod). Теперь вернемся к обычным методам экземпляра и посмотрим, как их можно "замаскировать" под обычные атрибуты.

В объектно-ориентированном программировании часто возникает ситуация: у нас есть какой-то атрибут, значение которого зависит от других атрибутов объекта.

#### Пример: класс Circle (Круг)

Давайте создадим класс для представления круга. У него будет один основной атрибут - radius (радиус). Но у круга есть и другие характеристики, например, diameter (диаметр) и area (площадь).

##### Традиционный подход с методами:

In [4]:
import math

class Circle:
    def __init__(self, radius):
        self.radius = radius
        
    def get_diameter(self):
        """Вычисляет и возвращает диаметр."""
        return self.radius * 2
        
    def get_area(self):
        """Вычисляет и возвращает площадь."""
        return math.pi * (self.radius ** 2)

# --- Как этим пользоваться? ---
c = Circle(10)
print(f"Радиус: {c.radius}")
print(f"Диаметр: {c.get_diameter()}") # <-- Вызов метода со скобками
print(f"Площадь: {c.get_area()}")   # <-- Вызов метода со скобками

Радиус: 10
Диаметр: 20
Площадь: 314.1592653589793


Это работает, но выглядит не очень красиво. Диаметр и площадь воспринимаются нами как <b>свойства</b> или <b>атрибуты</b> круга, а не как <b>действия</b>. Хотелось бы обращаться к ним так же просто, как к радиусу: c.diameter, а не c.get_diameter().

##### Элегантное решение с @property

Декоратор @property позволяет нам превратить метод экземляра в "вычисляемый атрибут". Мы пишем логику внутри метода, но снаружи можем обращаться к нему как к обычному атрибуту, <b>без круглых скобок ()</b>.

In [5]:
import math

class Circle:
    def __init__(self, radius):
        self.radius = radius
        
    @property
    def diameter(self):
        """Диаметр круга. Теперь это свойство."""
        print("-> (Вычисляю диаметр...)")
        return self.radius * 2
        
    @property
    def area(self):
        """Площадь круга. Теперь это свойство."""
        print("-> (Вычисляю площадь...)")
        return math.pi * (self.radius ** 2)

# --- Как этим пользоваться ТЕПЕРЬ? ---
c = Circle(10)
print(f"Радиус: {c.radius}")
print(f"Диаметр: {c.diameter}") # <-- Обращение как к атрибуту, БЕЗ ()
print(f"Площадь: {c.area:.2f}")     # <-- Обращение как к атрибуту, БЕЗ ()

# Если мы изменим радиус, свойства автоматически пересчитаются
c.radius = 12
print(f"\nНовый радиус: {c.radius}")
print(f"Новый диаметр: {c.diameter}")
print(f"Новая площадь: {c.area:.2f}")

Радиус: 10
-> (Вычисляю диаметр...)
Диаметр: 20
-> (Вычисляю площадь...)
Площадь: 314.16

Новый радиус: 12
-> (Вычисляю диаметр...)
Новый диаметр: 24
-> (Вычисляю площадь...)
Новая площадь: 452.39


<b>Анализ</b>:

- <b>Чистый интерфейс</b>: Код c.diameter гораздо более читаем и интуитивен, чем c.get_diameter(). Он отражает суть того, что диаметр - это свойство объекта.

- <b>"Ленивые" вычисления</b>: Обратите внимания на print внутри методов. Код для вычисления диаметра или площади выполняется только в тот момент, когда мы <b>обращаемся</b> к соответствующему свойству.

- <b>Только чтение (Read-only)</b>: В такой форме мы создали атрибуты, доступные только для чтения. Попытка присвоить им значение (c.diameter = 30) вызовет ошибку AttributeError.

#### Добавляем возможность записи: сеттеры (.setter)

@property также позволяет определить "сеттер" - метод, который будет вызываться при попытке присвоить значение нашему свойству. Это дает нам возможность добавить логику валидации.

In [6]:
class Person:
    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name

    @property
    def full_name(self):
        return f"{self.first_name} {self.last_name}"
    
    # Сеттер для свойства 'full_name'
    @full_name.setter
    def full_name(self, new_full_name):
        print("-> (Вызван сеттер для full_name!)")
        if " " not in new_full_name:
            raise ValueError("Требуется указать и имя, и фамилию.")
        # Разбираем строку и изменяем реальные атрибуты
        self.first_name, self.last_name = new_full_name.split(" ", 1)
        
p = Person("Иван", "Петров")
print(p.full_name) # Вызов геттера

p.full_name = "Анна Сидорова" # Вызов сеттера
print(p.first_name)
print(p.last_name)
print(p.full_name)

Иван Петров
-> (Вызван сеттер для full_name!)
Анна
Сидорова
Анна Сидорова


#### Итог

Декоратор @property - это мощный инструмент для создания чистого и интуитивного интефейса для ваших классов. Он позволяет:

1. Представить вычисляеме значения как простые атрибуты (геттеры).

2. Скрыть внутреннюю реализацию и предоставить пользователю класса простой доступ к данным.

3. При необходимости добавить логику валидации при изменении этих "вирутальных" атрибутов (сеттеры).

### Задачи

#### Задача 1: Статический метод-утилита

<b>Условие задачи</b>:

Вам дан класс Validator. Напишите внутри него <b>статический</b> метод is_valid_email(email_string).

Этот метод должен проверять, содержит ли строка email_string символ @. Он должен возвращать True, если содержит, и False в противном случае.

Так как для этой проверки не нужно состояние экземпляра, staticmethod — идеальный выбор.

In [8]:
import re

class Validator:
    @staticmethod
    def is_valid_email(email_string):
        pattern = r'.+@\w+.\w{2,}'
        return bool(re.match(pattern, email_string))

#### Задача 2: Классовый метод как альтернативынй конструктор

<b>Условие задачи</b>:

Вам дан класс Order с конструктором \_\_init__(self, order_id).

Напишите <b>классовый</b> метод from_string(cls, order_string), который будет работать как альтернативный конструктор.

Этот метод должен принимать строку вида "order:123", извлекать из нее ID заказа и создавать новый экземпляр класса Order с этим ID.

In [9]:
class Order:
    def __init__(self, order_id):
        self.order_id = order_id

    @classmethod
    def from_string(cls, order_string):
        order_id = order_string[order_string.index(':') + 1:]
        return cls(order_id)

#### Задача 3: Вычисляемое свойство (property)

<b>Условие задачи</b>:

Вам дан класс Person с атрибутами first_name и last_name.

Создайте <b>свойство (property)</b> full_name, которое будет возвращать полное имя человека, состоящее из имени и фамилии, разделенных пробелом.

Обращаться к full_name нужно будет как к атрибуту, без скобок.

In [10]:
class Person:
    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name

    @property
    def full_name(self):
        return f'{self.first_name} {self.last_name}'

#### Задача 4: Свойство с сеттером

<b>Условие задачи</b>:

Вам дан класс BankAccount с "приватным" атрибутом _balance.
Создайте свойство balance, которое:
1. При чтении (@property) возвращает значение _balance.

2. При попытке присвоения (@balance.setter) проверяет, что новое значение является неотрицательным числом (>= 0). Если это не так, сеттер должен выбросить ValueError с сообщением "Invalid balance".

In [11]:
class BankAccount:
    def __init__(self, initial_balance):
        self._balance = initial_balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, new_balance):
        if new_balance < 0:
            raise ValueError('Invalid balance')
        self._balance = new_balance

#### Задача 5: Выбери правильный декоратор

<b>Условие задачи</b>:

Вам дан класс System. В нем есть три заготовки методов. Примените к каждому из них <b>правильный</b> встроенный декоратор (@staticmethod, @classmethod, @property) в соответствии с комментариями-требованиями.

In [12]:
class System:
    def __init__(self, status):
        self.status = status

    # ТРЕБОВАНИЕ: Этот метод должен быть статическим.
    # Он не использует ни self, ни cls.
    @staticmethod
    def get_version():
        return "1.0.0"

    # ТРЕБОВАНИЕ: Этот метод должен быть классовым.
    # Он должен создавать экземпляр класса со статусом 'default'.
    @classmethod
    def default_system(cls):
        return cls("default")

    # ТРЕБОВАНИЕ: Это должно быть свойство.
    # Оно должно возвращать True, если self.status == 'active'.
    @property
    def is_active(self):
        return self.status == 'active'